# 🏥 Diagnostic Adapter — Fine-tuning

Trains the model to produce a **structured differential diagnosis** from patient symptoms.

**Primary dataset: MedQA USMLE** — clinical vignettes written by medical professionals,
specifically designed to test differential diagnosis reasoning.

**Supplementary: DDxPlus** — adds probability-ranked differential structure.

**Output format the model learns (matches runtime prompt exactly):**
```
1. Most likely condition: [name] — [reason]
2. Other conditions to consider: [name] — [why less likely]
3. Recommended next steps: [tests / specialist / urgency]
```

**Output:** `diagnostic_adapter/` folder saved to Google Drive


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new]', 'peft', 'accelerate',
    'bitsandbytes', 'trl', 'datasets', 'huggingface_hub', '-q'
], check=True)
print('✅ Installed.')

In [ ]:
# ── Mount Drive & set paths ───────────────────────────────────────────────────
from google.colab import drive
import os, json, random, glob, gc
import torch

drive.mount('/content/drive', force_remount=False)

# WHERE THE ADAPTER WILL BE SAVED
SAVE_PATH  = '/content/drive/MyDrive/unsloth_training/diagnostic_adapter'
CKPT_DIR   = '/content/drive/MyDrive/unsloth_training/diagnostic_checkpoints'
CACHE_PATH = '/content/drive/MyDrive/unsloth_training/diagnostic_dataset.json'

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

# ── Model config ──────────────────────────────────────────────────────────────
BASE_MODEL     = 'unsloth/llama-2-7b-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

# ── LoRA config (must match follow_up_adapter exactly) ────────────────────────
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0
TARGET_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

# ── Training config ───────────────────────────────────────────────────────────
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS     = 1
LR         = 2e-4
SEED       = 3407
SAVE_STEPS = 100

# ── Dataset caps ──────────────────────────────────────────────────────────────
MAX_USMLE   = 10_000   # MedQA USMLE train set is ~10k — use all of it
MAX_DDXPLUS = 25_000   # DDxPlus is 1.3M — cap to keep compute balanced

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Config ready. GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'   Adapter will be saved to: {SAVE_PATH}')

In [ ]:
# ── Output format helpers ─────────────────────────────────────────────────────
#
# CRITICAL: The format here must match the prompt in medical_agent_pipeline.ipynb
# so that what the model learns at training = what it's asked to produce at runtime.
#
SYSTEM_PROMPT = (
    'You are an experienced medical diagnostic assistant. '
    'Given a patient clinical presentation, provide a structured assessment.'
)

FORMAT_GUIDE = (
    'Structure your response exactly as:\n'
    '1. Most likely condition: [name] — [brief reason]\n'
    '2. Other conditions to consider: [name] — [why less likely] (list 2–3)\n'
    '3. Recommended next steps: [tests / referral / urgency level]'
)

def make_instruction(case_text: str) -> str:
    return (
        f'{SYSTEM_PROMPT}\n\n'
        f'Clinical case:\n{case_text.strip()}\n\n'
        f'{FORMAT_GUIDE}'
    )

def to_sft(instruction: str, response: str) -> str:
    return f'### Instruction:\n{instruction}\n\n### Response:\n{response}'

print('✅ Helpers defined.')

In [ ]:
# ── MedQA USMLE converter ─────────────────────────────────────────────────────
#
# Dataset columns:
#   'question'  : clinical vignette (free text)
#   'options'   : {'A': '...', 'B': '...', 'C': '...', 'D': '...'}
#   'answer'    : correct key  e.g. 'C'  (NOT the answer text)
#
def convert_usmle(row: dict) -> dict | None:
    question   = (row.get('question') or '').strip()
    options    = row.get('options') or {}
    answer_key = (row.get('answer') or '').strip().upper()

    if not question or not options or answer_key not in options:
        return None

    correct     = options[answer_key].strip()
    distractors = [v.strip() for k, v in options.items() if k != answer_key]

    # Build differential section from wrong options
    diff_lines = '\n'.join(
        f'   {chr(97+i)}. {d} — less consistent with the complete clinical picture'
        for i, d in enumerate(distractors[:3])
    )

    response = (
        f'1. Most likely condition: {correct}\n'
        f'   Reasoning: The clinical presentation — including the specific symptom '
        f'pattern, patient demographics, and history described — is most consistent '
        f'with {correct}. This diagnosis best explains the full picture.\n\n'
        f'2. Other conditions to consider:\n{diff_lines}\n\n'
        f'3. Recommended next steps: Perform targeted diagnostic workup to confirm '
        f'{correct}. This includes relevant laboratory tests, imaging studies, or '
        f'specialist referral based on clinical severity and urgency.'
    )

    return {'text': to_sft(make_instruction(question), response)}


print('✅ USMLE converter defined.')

In [ ]:
# ── DDxPlus converter ─────────────────────────────────────────────────────────
#
# Dataset columns:
#   'PATHOLOGY'              : ground-truth diagnosis
#   'EVIDENCES'              : list of symptom codes e.g. ['E_53', 'E_54_@_V_181']
#   'DIFFERENTIAL_DIAGNOSIS' : [[condition, probability], ...]
#   'AGE', 'SEX'
#
def convert_ddxplus(row: dict, evidence_map: dict) -> dict | None:
    pathology = (row.get('PATHOLOGY') or '').strip()
    evidences = row.get('EVIDENCES') or []
    ddx       = row.get('DIFFERENTIAL_DIAGNOSIS') or []
    age       = row.get('AGE', '?')
    sex       = row.get('SEX', '?')

    if not pathology or not evidences:
        return None

    # Decode symptom codes → readable text
    symptoms = []
    for ev in evidences:
        base = ev.split('_@_')[0]
        symptoms.append(evidence_map.get(ev) or evidence_map.get(base) or ev)

    case_text = (
        f'Patient: {sex}, age {age}\n'
        f'Presenting symptoms: {", ".join(symptoms)}'
    )

    # Sort differential by probability
    ddx_sorted = sorted(ddx, key=lambda x: float(x[1]), reverse=True)
    top_prob   = round(float(ddx_sorted[0][1]) * 100, 1) if ddx_sorted else 0

    alt_lines = '\n'.join(
        f'   {chr(97+i)}. {cond} ({round(float(p)*100,1)}%) — '
        f'shares some features but lower probability given full symptom set'
        for i, (cond, p) in enumerate(ddx_sorted[1:4])
        if cond != pathology
    ) or '   (no significant alternatives)'

    response = (
        f'1. Most likely condition: {pathology} ({top_prob}% probability)\n'
        f'   Reasoning: Given the symptom profile — {", ".join(symptoms[:4])} — '
        f'{pathology} is the most consistent diagnosis.\n\n'
        f'2. Other conditions to consider:\n{alt_lines}\n\n'
        f'3. Recommended next steps: Thorough clinical examination and appropriate '
        f'diagnostic workup to confirm {pathology}. Urgency depends on symptom '
        f'severity and acuity.'
    )

    return {'text': to_sft(make_instruction(case_text), response)}


print('✅ DDxPlus converter defined.')

In [ ]:
# ── Load & preprocess datasets ────────────────────────────────────────────────
from datasets import load_dataset, Dataset

if os.path.exists(CACHE_PATH):
    print(f'📂 Loading cached dataset...')
    with open(CACHE_PATH) as f:
        all_examples = json.load(f)
    print(f'   ✅ {len(all_examples):,} examples loaded from cache.')

else:
    all_examples = []

    # ── SOURCE 1: MedQA USMLE (primary) ──────────────────────────────────────
    print('⏳ Loading MedQA USMLE (GBaker/MedQA-USMLE-4-options)...')
    usmle_ex = []
    try:
        ds = load_dataset('GBaker/MedQA-USMLE-4-options', trust_remote_code=True)
        for row in ds['train']:
            if len(usmle_ex) >= MAX_USMLE:
                break
            ex = convert_usmle(row)
            if ex:
                usmle_ex.append(ex)
        all_examples.extend(usmle_ex)
        print(f'   ✅ MedQA USMLE → {len(usmle_ex):,} examples')
    except Exception as e:
        print(f'   ⚠️  MedQA USMLE failed: {e}')

    # ── SOURCE 2: DDxPlus (ranked differential) ───────────────────────────────
    print('\n⏳ Loading DDxPlus...')
    ddxplus_ex  = []
    evidence_map = {}
    try:
        # Load dataset
        try:
            ddx_ds = load_dataset('mcrmi/ddxplus', trust_remote_code=True)
        except Exception:
            ddx_ds = load_dataset('aai530-group6/ddxplus', trust_remote_code=True)

        # Try to load evidence metadata (symptom code → readable text)
        try:
            from huggingface_hub import hf_hub_download
            ev_path = hf_hub_download(
                repo_id='mcrmi/ddxplus',
                filename='release_evidences.json',
                repo_type='dataset'
            )
            with open(ev_path) as f:
                ev_meta = json.load(f)
            evidence_map = {k: v.get('question_en', k) for k, v in ev_meta.items()}
            print(f'   Evidence map: {len(evidence_map):,} codes decoded')
        except Exception:
            print('   ℹ️  Evidence metadata not found — using raw codes')

        ddx_split = ddx_ds.get('train', ddx_ds[list(ddx_ds.keys())[0]])
        for row in ddx_split:
            if len(ddxplus_ex) >= MAX_DDXPLUS:
                break
            ex = convert_ddxplus(row, evidence_map)
            if ex:
                ddxplus_ex.append(ex)
        all_examples.extend(ddxplus_ex)
        print(f'   ✅ DDxPlus     → {len(ddxplus_ex):,} examples')
    except Exception as e:
        print(f'   ⚠️  DDxPlus failed: {e}')

    # ── Shuffle & cache ───────────────────────────────────────────────────────
    random.seed(SEED)
    random.shuffle(all_examples)

    with open(CACHE_PATH, 'w') as f:
        json.dump(all_examples, f, ensure_ascii=False)
    print(f'\n💾 Cached → {CACHE_PATH}')

print(f'\n📊 Dataset summary:')
print(f'   MedQA USMLE : {len([x for x in all_examples if "Most likely" in x["text"]]):>7,}  (estimated)')
print(f'   TOTAL       : {len(all_examples):>7,}')

# Show one sample
print('\n── Sample training example (USMLE) ──')
print(all_examples[0]['text'][:800])

hf_dataset = Dataset.from_list(all_examples)
print(f'\n✅ HuggingFace Dataset: {len(hf_dataset):,} rows')

In [ ]:
# ── Load base model + attach LoRA ─────────────────────────────────────────────
from unsloth import FastLanguageModel

print('⏳ Loading base model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = LORA_R,
    target_modules            = TARGET_MODULES,
    lora_alpha                = LORA_ALPHA,
    lora_dropout              = LORA_DROPOUT,
    bias                      = 'none',
    use_gradient_checkpointing= 'unsloth',
    random_state              = SEED,
)
tokenizer.padding_side = 'right'

tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model ready. Trainable params: {tp/1e6:.2f}M')

In [ ]:
# ── Resume from checkpoint (optional) ────────────────────────────────────────
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')),
               key=lambda p: int(p.split('-')[-1]))
RESUME_FROM = ckpts[-1] if ckpts else None

if RESUME_FROM:
    print(f'♻️  Resuming from: {RESUME_FROM}')
else:
    print('🆕 Training from scratch.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = hf_dataset,
    args          = SFTConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = 5,
        num_train_epochs            = EPOCHS,
        learning_rate               = LR,
        lr_scheduler_type           = 'linear',
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        logging_steps               = 20,
        output_dir                  = CKPT_DIR,
        save_strategy               = 'steps',
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        report_to                   = 'none',
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LENGTH,
        packing                     = False,
        seed                        = SEED,
    ),
)

print(f'🚀 Training diagnostic_adapter on {len(hf_dataset):,} examples...')
result = trainer.train(resume_from_checkpoint=RESUME_FROM)
print(f'\n✅ Done! Loss: {result.training_loss:.4f} | Steps: {result.global_step}')

In [ ]:
# ── Save adapter to Drive ─────────────────────────────────────────────────────
print(f'💾 Saving diagnostic_adapter to: {SAVE_PATH}')
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print('\n✅ Saved files:')
for f in sorted(os.listdir(SAVE_PATH)):
    sz = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f'   {f:45s}  {sz:.1f} MB')
print(f'\n📁 Full path: {SAVE_PATH}')

In [ ]:
# ── Format test ───────────────────────────────────────────────────────────────
# Verifies the adapter produces the expected 3-section structured output
FastLanguageModel.for_inference(model)

TEST_CASE = (
    'A 52-year-old male smoker with hypertension presents with sudden onset '
    'severe chest pain radiating to his left arm and jaw, diaphoresis, and '
    'shortness of breath for 45 minutes. BP 160/95, HR 98, SpO2 94%. '
    'ECG shows ST elevation in leads II, III, and aVF.'
)

prompt = f'### Instruction:\n{make_instruction(TEST_CASE)}\n\n### Response:\n'
inputs = tokenizer([prompt], return_tensors='pt').to(DEVICE)

with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=350, temperature=0.7, top_p=0.9,
        do_sample=True, repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

new      = out[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(new, skip_special_tokens=True).strip()

print('='*70)
print('🔬 TEST CASE:')
print(TEST_CASE)
print('\n📋 MODEL RESPONSE:')
print(response)
print('='*70)

# Format check
checks = {
    'Section 1 — Most likely condition' : '1.' in response,
    'Section 2 — Differential'          : '2.' in response,
    'Section 3 — Next steps'            : '3.' in response,
}
print('\n✅ Format check:')
for label, passed in checks.items():
    print(f'   {"✅" if passed else "❌"}  {label}')

if all(checks.values()):
    print('\n🎉 All sections present — adapter output format is correct!')
else:
    print('\n⚠️  Some sections missing — model may need more training steps.')

In [ ]:
# ── Free VRAM ─────────────────────────────────────────────────────────────────
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print('🧹 VRAM freed.')
print(f'\n📁 Adapter saved at: {SAVE_PATH}')
print('Next: run medical_agent_pipeline.ipynb to launch the full agent.')